In [54]:
import itertools
import pandas as pd

file_id = '1xJZO4bt5VZf4Ix-w-QPGMVpPxjlCQac3'
url = f'https://drive.google.com/uc?export=download&id={file_id}'
base = pd.read_excel(url)

valores = base['valor'].tolist()
valor_esperado = 4886.85

valores_int = [int(round(v * 100)) for v in valores]
valor_esperado_int = int(round(valor_esperado * 100))
total_int = sum(valores_int)
n = len(valores_int)
combinacoes_possiveis = 2**len(valores) - 1 - len(valores)

print("Total:", total_int / 100)
print("Valor Esperado:", valor_esperado)
print("Total Combinações Possíveis:", combinacoes_possiveis)

meio = n // 2
esquerda = list(range(meio))
direita = list(range(meio, n))

mapa = {}
for r in range(len(esquerda) + 1):
    for combo in itertools.combinations(esquerda, r):
        s = sum(valores_int[i] for i in combo)
        mapa.setdefault(s, []).append(combo)

solucoes = []
parar = False
for r in range(len(direita) + 1):
    for combo_dir in itertools.combinations(direita, r):
        s = sum(valores_int[i] for i in combo_dir)
        faltando = valor_esperado_int - s
        if faltando in mapa:
            for combo_esq in mapa[faltando]:
                solucao = [0] * n
                for i in combo_esq:
                    solucao[i] = 1
                for i in combo_dir:
                    solucao[i] = 1
                solucoes.append(solucao)
                if len(solucoes) >= combinacoes_possiveis:
                    parar = True
                    break
        if parar:
            break
    if parar:
        break

txt = "solução encontrada" if len(solucoes) == 1 else "soluções encontradas"

print(f"\n{len(solucoes)} {txt}:\n")
for idx, solucao in enumerate(solucoes):

    indices_mantidos = [i for i in range(len(valores)) if solucao[i] == 1]

    mantidos = [(i, valores[i]) for i in indices_mantidos]
    base_mantida = base.loc[indices_mantidos]
    print(f"Solução {idx + 1}")

    for i, valor in mantidos:
        print(f"manter {valor} | índice {i}")
    print("\nCategoria:", base_mantida['categoria'].unique())
    print("\nData:", base_mantida['data_compra'].unique())
    print("\nVendedor:", base_mantida['vendedor'].unique())

    print("\nProduto:", base_mantida['produto'].unique())
    print("\nCliente:", base_mantida['tipo_cliente'].unique())
    print("\nValores únicos por coluna:")
    for col in ['categoria','produto','data_compra','vendedor','tipo_cliente']:
        print(f"{col}: {base_mantida[col].nunique()}")
    print("-" * 40)

Total: 16842.65
Valor Esperado: 4886.85
Total Combinações Possíveis: 268435427

93 soluções encontradas:

Solução 1
manter 110.33 | índice 1
manter 729.62 | índice 2
manter 909.05 | índice 3
manter 931.49 | índice 6
manter 461.14 | índice 8
manter 487.73 | índice 10
manter 960.31 | índice 11
manter 297.18 | índice 21

Categoria: ['Casa' 'Esportes' 'Eletrônicos']

Data: ['abr/26' 'fev/25' 'jan/25' 'junho/26' 'mai/25' 'mai/26']

Vendedor: ['João Silva' 'Carlos Oliveira']

Produto: ['Sofá' 'Bicicleta' 'Tablet' 'Celular' 'Microondas']

Cliente: ['Novo' 'Recorrente']

Valores únicos por coluna:
categoria: 3
produto: 5
data_compra: 6
vendedor: 2
tipo_cliente: 2
----------------------------------------
Solução 2
manter 703.13 | índice 0
manter 749.95 | índice 4
manter 791.47 | índice 5
manter 931.49 | índice 6
manter 597.28 | índice 9
manter 487.73 | índice 10
manter 422.85 | índice 15
manter 202.95 | índice 17

Categoria: ['Moda' 'Eletrônicos' 'Casa']

Data: ['abr/25' 'fev/26' 'jan/25' 'junh

In [81]:
import pandas as pd

n_sol = len(solucoes)
colunas = ['vendedor', 'categoria', 'data_compra', 'produto', 'tipo_cliente']

print(f"Analisando {n_sol} soluções\n")
for col in colunas:
    print(f"=== {col} ===")
    freq = {}
    for solucao in solucoes:
        idx = [i for i in range(len(solucao)) if solucao[i] == 1]
        for valor in base.loc[idx, col].unique():
            freq[valor] = freq.get(valor, 0) + 1

    linhas = []
    for valor, qtd in freq.items():
        p = qtd / n_sol
        poder = 1 - 2 * abs(p - 0.5)
        linhas.append((valor, qtd, p, poder))

    linhas.sort(key=lambda x: -x[3])
    for valor, qtd, p, poder in linhas:
        tag = "discrimina" if poder > 0.6 else ("estrutural" if poder < 0.15 else "")
        print(f"  {str(valor):20s} {qtd:3d}/{n_sol} ({p:4.0%})  poder={poder:.2f} {tag}")
    print()

Analisando 93 soluções

=== vendedor ===
  Maria Souza           26/93 ( 28%)  poder=0.56 
  Carlos Oliveira       75/93 ( 81%)  poder=0.39 
  João Silva            93/93 (100%)  poder=0.00 estrutural

=== categoria ===
  Moda                  37/93 ( 40%)  poder=0.80 discrimina
  Esportes              78/93 ( 84%)  poder=0.32 
  Casa                  87/93 ( 94%)  poder=0.13 estrutural
  Eletrônicos           93/93 (100%)  poder=0.00 estrutural

=== data_compra ===
  mai/25                39/93 ( 42%)  poder=0.84 discrimina
  jan/25                37/93 ( 40%)  poder=0.80 discrimina
  abr/26                61/93 ( 66%)  poder=0.69 discrimina
  mar/25                30/93 ( 32%)  poder=0.65 discrimina
  fev/26                29/93 ( 31%)  poder=0.62 discrimina
  junho/26              65/93 ( 70%)  poder=0.60 discrimina
  abr/25                22/93 ( 24%)  poder=0.47 
  jun/25                20/93 ( 22%)  poder=0.43 
  fev/25                18/93 ( 19%)  poder=0.39 
  mai/26           

In [79]:
import pandas as pd
from itertools import combinations

n_sol = len(solucoes)
cols = ['vendedor', 'categoria', 'data_compra', 'tipo_cliente', 'produto']

perfis = []
for solucao in solucoes:
    idx = [i for i in range(len(solucao)) if solucao[i] == 1]
    perfil = {col: frozenset(base.loc[idx, col].unique()) for col in cols}
    perfil['_n_itens'] = len(idx)
    perfis.append(perfil)

print(f"{n_sol} soluções enumeradas.\n")

print("=== Soluções mais concentradas (poucos valores distintos) ===")
def concentracao(p):
    return sum(len(p[c]) for c in cols)   # soma de cardinalidades
ranking = sorted(range(n_sol), key=lambda k: concentracao(perfis[k]))
for k in ranking[:5]:
    p = perfis[k]
    resumo = ", ".join(f"{c}={len(p[c])}" for c in cols)
    print(f"  Solução {k+1}: {resumo}")

print("\n=== Combinações (coluna × valor) mais raras entre as soluções ===")
from collections import Counter
par_freq = Counter()
for p in perfis:
    for c1, c2 in combinations(cols, 2):
        for v1 in p[c1]:
            for v2 in p[c2]:
                par_freq[(c1, v1, c2, v2)] += 1
raros = sorted(par_freq.items(), key=lambda x: x[1])
for (c1, v1, c2, v2), q in raros[:10]:
    print(f"  {v1} ({c1}) + {v2} ({c2}): {q}/{n_sol} soluções")

print("")

PISTAS = {}

def add_pista(*args):
    if len(args) == 1 and isinstance(args[0][0], (list, tuple)):
        pistas = args[0]            # add_pista([p1, p2, ...])  -> lista de pistas
    elif len(args) == 1:
        pistas = [args[0]]          # add_pista([nome, col, regra]) -> uma pista
    else:
        pistas = [args]             # add_pista(nome, col, regra)   -> uma pista (3 soltos)

    for nome, coluna, regra in pistas:
        PISTAS[nome] = (coluna, regra)
        print(f"pista '{nome}' adicionada (coluna: {coluna}).")
    print(f"\nCatálogo: {list(PISTAS)}\n")

def analisar(*nomes_pistas):
    sobreviventes = []
    for k, p in enumerate(perfis):
        if all(PISTAS[nome][1](p[PISTAS[nome][0]]) for nome in nomes_pistas):
            sobreviventes.append(k + 1)
    print(f"{list(nomes_pistas)}: {len(sobreviventes)} soluções -> {sobreviventes}")

add_pista([
    ["quantidade_categoria", "categoria", lambda s: len(s) == 3],
    ["quantidade_vendedores", "vendedor", lambda s: len(s) == 2],
    ["quantidade_data", "data_compra", lambda s: len(s) == 4],
    ["quantidade_produtos", "produto", lambda s: len(s) == 7]
])

analisar("quantidade_categoria","quantidade_vendedores","quantidade_data")

93 soluções enumeradas.

=== Soluções mais concentradas (poucos valores distintos) ===
  Solução 11: vendedor=2, categoria=2, data_compra=4, tipo_cliente=2, produto=4
  Solução 37: vendedor=1, categoria=2, data_compra=4, tipo_cliente=2, produto=5
  Solução 40: vendedor=1, categoria=2, data_compra=3, tipo_cliente=2, produto=6
  Solução 49: vendedor=1, categoria=3, data_compra=2, tipo_cliente=2, produto=6
  Solução 68: vendedor=1, categoria=2, data_compra=2, tipo_cliente=2, produto=7

=== Combinações (coluna × valor) mais raras entre as soluções ===
  Maria Souza (vendedor) + jun/25 (data_compra): 1/93 soluções
  Maria Souza (vendedor) + Calça (produto): 1/93 soluções
  Moda (categoria) + fev/25 (data_compra): 2/93 soluções
  fev/25 (data_compra) + Camiseta (produto): 2/93 soluções
  jun/25 (data_compra) + Panela (produto): 2/93 soluções
  fev/26 (data_compra) + Bicicleta (produto): 3/93 soluções
  Maria Souza (vendedor) + AirFryer (produto): 3/93 soluções
  mai/25 (data_compra) + Calça 